In [1]:
!pip install timm

import torch
import torch.nn as nn
import torch.optim as optim
import timm
import numpy as np
import os
import cv2
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

In [2]:
DATASET_PATH = "/kaggle/input/datasets/sekhsujonhaque/cape-gooseberry-classification-dataset/classification_dataset"

In [3]:
class HSVTransform:
    def __call__(self, img):
        img = np.array(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        return Image.fromarray(img)

In [14]:
train_transforms = transforms.Compose([
    HSVTransform(),
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor()
])

val_transforms = transforms.Compose([
    HSVTransform(),
    transforms.Resize((300, 300)),
    transforms.ToTensor()
])

In [15]:
train_dataset = datasets.ImageFolder(DATASET_PATH + "/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATASET_PATH + "/valid", transform=val_transforms)
test_dataset  = datasets.ImageFolder(DATASET_PATH + "/test", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print("Classes:", train_dataset.classes)

Classes: ['INTERMEDIA', 'MADURO', 'NO_APTA', 'VERDE']


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [17]:
labels = [label for _, label in train_dataset]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [18]:
model = timm.create_model('efficientnet_b3', pretrained=True)

# Freeze base
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last layers
for param in model.blocks[-5:].parameters():
    param.requires_grad = True

# Custom classifier
model.classifier = nn.Sequential(
    nn.BatchNorm1d(model.classifier.in_features),
    nn.Dropout(0.4),
    nn.Linear(model.classifier.in_features, 4)
)

model = model.to(device)

In [19]:
optimizer = optim.AdamW(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [20]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [21]:
EPOCHS = 35
best_val = 0
patience = 5
counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    val_acc = evaluate(val_loader)

    if val_acc > best_val:
        best_val = val_acc
        counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_model_hsv.pth")
        print("Best model saved")
    else:
        counter += 1

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if counter >= patience:
        print("Early stopping triggered")
        break

Best model saved
Epoch 1/35, Loss: 66.1336, Val Acc: 35.67%
Best model saved
Epoch 2/35, Loss: 55.1743, Val Acc: 40.94%
Best model saved
Epoch 3/35, Loss: 47.8911, Val Acc: 52.63%
Best model saved
Epoch 4/35, Loss: 42.5494, Val Acc: 55.56%
Best model saved
Epoch 5/35, Loss: 38.0204, Val Acc: 59.65%
Best model saved
Epoch 6/35, Loss: 31.7806, Val Acc: 61.99%
Epoch 7/35, Loss: 31.3183, Val Acc: 60.23%
Best model saved
Epoch 8/35, Loss: 30.3394, Val Acc: 62.57%
Best model saved
Epoch 9/35, Loss: 27.1066, Val Acc: 64.33%
Epoch 10/35, Loss: 26.2120, Val Acc: 63.74%
Epoch 11/35, Loss: 23.8989, Val Acc: 64.33%
Best model saved
Epoch 12/35, Loss: 23.1747, Val Acc: 64.91%
Epoch 13/35, Loss: 22.3941, Val Acc: 64.33%
Epoch 14/35, Loss: 21.6430, Val Acc: 63.16%
Epoch 15/35, Loss: 20.8433, Val Acc: 64.33%
Epoch 16/35, Loss: 20.1453, Val Acc: 64.91%
Epoch 17/35, Loss: 20.1470, Val Acc: 63.16%
Early stopping triggered


In [22]:
model.load_state_dict(torch.load("/kaggle/working/best_model_hsv.pth"))
model.eval()

test_acc = evaluate(test_loader)
print("FINAL HSV TEST ACCURACY:", test_acc)

FINAL HSV TEST ACCURACY: 61.224489795918366
